## LIBRARIES AND DECLARATION

In [1]:
import pandas as pd
import json
import re
from IPython.display import FileLink

## CONFIG

In [2]:
STAGE = 3  # Stage 1: 3 attempts | Stage 2: 2 attempts | Stage 3: 2/3 attempts

CONFIG = {
    (1,): {
        "files": [
            "/kaggle/input/private-dataset/step2_logic1_llama-3.3-70b-versatile_1.csv",
            "/kaggle/input/private-dataset/step2_logic1_llama-3.3-70b-versatile_2.csv",
            "/kaggle/input/private-dataset/step2_logic1_llama-3.3-70b-versatile_3.csv",
        ],
        "out": "step1_classify1_logic1_llama.csv",
    },
    (2, 3): { 
        "files": [
            "/kaggle/input/datasets/dhuyent/stage3/step2_filter1_reference1_qwen3.7-max_1.csv", 
            "/kaggle/input/datasets/dhuyent/stage3/step2_filter1_reference1_qwen3.7-max_2.csv",
            "/kaggle/input/datasets/dhuyent/stage3/step2_filter1_reference1_qwen3.7-max_3.csv"
        ],
        "out": "classify3_reference1_qwen3.7-max.csv",
    },
}

def get_config(stage):
    for keys, cfg in CONFIG.items():
        if stage in keys:
            return cfg
    raise KeyError(f"No config for stage {stage}")

CFG         = get_config(STAGE)
FILES       = CFG["files"]
OUT_PATH    = CFG["out"]
N_ATTEMPTS  = len(FILES)  # số lần chạy = số file
print(f"STAGE {STAGE}: {N_ATTEMPTS} attempts -> {OUT_PATH}")

STAGE 3: 3 attempts -> classify3_reference1_qwen3.7-max.csv


## LOAD DATA

In [3]:
df = pd.concat([pd.read_csv(f) for f in FILES], ignore_index=False)
print("Number of ids:", df.shape[0])
print("Unique ids:", df["id"].nunique())
df.head()

Number of ids: 153
Unique ids: 51


,topic,level,id,problem,solution,url,buggy_submission,bug_description,gt_status,gt_input,gt_actual_output,gt_expected_output,gt_reason,llm_model,prompt_strategy,pred_status,pred_input,pred_actual_output,pred_expected_output,pred_reason
0,String,Medium,18,"The string ""PAYPALISHIRING"" is written in a zi...","#pragma GCC optimize(""Ofast"")\n\n#include <ios...",https://leetcode.com/problems/zigzag-conversion,"#pragma GCC optimize(""Ofast"")\n\n#include <ios...",[Reference – Off-by-one index] Changed the mai...,Accepted,NaN,NaN,NaN,NaN,qwen3.7-max,zero-shot,Runtime Error,"""s = \""ABC\"", numRows = 2""","""Runtime Error""","""\""ACB\""""",The loop condition `i <= s.size()` causes out-...
1,String,Medium,20,The count-and-say sequence is a sequence of di...,class Solution {\npublic:\n string countAnd...,https://leetcode.com/problems/count-and-say,class Solution {\npublic:\n string countAnd...,[Reference – Off-by-one index] Changed the loo...,Wrong Answer,n =\n11,"""111312211331121321132122211\u0000""","""11131221133112132113212221""",NaN,qwen3.7-max,zero-shot,Wrong Answer,"""n = 2""","""111""","""11""",NaN
2,String,Medium,21,You have intercepted a secret message encoded ...,"#pragma GCC optimize(""Ofast"")\n\n#include <ios...",https://leetcode.com/problems/decode-ways,"#pragma GCC optimize(""Ofast"")\n\n#include <ios...",[Reference – Off-by-one index] Changed the DP ...,Runtime Error,NaN,NaN,NaN,Line 52: Char 21: runtime error: index 2 out o...,qwen3.7-max,zero-shot,Wrong Answer,"""s = \""110\""""","""2""","""1""",The code incorrectly handles cases where the c...
3,String,Medium,23,"Given a string s, find the length of the longe...",class Solution {\n public:\n int lengthOfLong...,https://leetcode.com/problems/longest-substrin...,class Solution {\n public:\n int lengthOfLong...,[Reference – Off-by-one index] Changed the rig...,Wrong Answer,"s =\n""bbbbb""",2,1,NaN,qwen3.7-max,zero-shot,Wrong Answer,"""s = \""abc\""""","""4""","""3""",NaN
4,String,Medium,29,"Given an array of strings strs, group the anag...",class Solution {\n public:\n vector<vector<st...,https://leetcode.com/problems/group-anagrams,class Solution {\n public:\n vector<vector<st...,[Reference – Off-by-one index] Changed the sor...,Wrong Answer,"strs = \n[""cab"",""tin"",""pew"",""duh"",""may"",""ill"",...","[[""doc""],[""ill""],[""duh""],[""may"",""max""],[""pew""]...","[[""max""],[""buy""],[""doc""],[""may""],[""ill""],[""duh...",NaN,qwen3.7-max,zero-shot,Runtime Error,"""strs = [\""eat\"",\""tea\"",\""tan\"",\""ate\"",\""nat...","""Runtime Error""","""[[\""bat\""],[\""nat\"",\""tan\""],[\""ate\"",\""eat\""...",Using end(key) + 1 in std::sort creates an out...


## LABEL CLASSIFICATION

Classify each `id` based on how well `pred_status` (LLM prediction) matches `gt_status` (ground truth) across multiple runs (attempts).

- **Label 1**: ALL attempts match (`pred_status == gt_status`)
- **Label 2**: SOME but not all attempts match (partial match)
- **Label 3**: NO attempt matches

In [4]:
# So sánh gt_status vs pred_status
df["match"] = df.apply(
    lambda r: "sim" if r["gt_status"] == r["pred_status"] else "diff",
    axis=1
)

# Group theo id, đếm sim - diff trong mỗi nhóm
group_stats = (
    df.groupby("id")["match"]
    .value_counts()
    .unstack(fill_value=0)
    .rename(columns={"sim": "n_sim", "diff": "n_diff"})
    .reset_index()
)

# Đảm bảo cả hai cột luôn tồn tại dù tất cả là sim hoặc diff
for col in ("n_sim", "n_diff"):
    if col not in group_stats.columns:
        group_stats[col] = 0

In [5]:
# Check if ids whose `pred_status` differs on EVERY attempt
uniq = df.groupby("id")["pred_status"].nunique().reset_index(name="n_unique")
cnt  = df.groupby("id").size().reset_index(name="n_rows")
uniq = uniq.merge(cnt, on="id")

ids_all_diff = uniq.loc[
    (uniq["n_rows"] == N_ATTEMPTS) & (uniq["n_unique"] == N_ATTEMPTS),
    "id",
]

print("Number of ids with a different pred_status on every attempt:", len(ids_all_diff))
print(ids_all_diff.tolist())

Number of ids with a different pred_status on every attempt: 1
[35]


In [6]:
def assign_label(row, n_attempts=N_ATTEMPTS):
    s, d = row["n_sim"], row["n_diff"]
    total = s + d
    if total != n_attempts:
        return None          # group has missing/extra rows: flag for inspection
    if s == total:           # all sim
        return 1
    if d == total:           # all diff
        return 3
    return 2                 # partial match

group_stats["label"] = group_stats.apply(assign_label, axis=1)

df = df.drop(columns=["label"], errors="ignore")
df = df.merge(group_stats[["id", "label"]], on="id", how="left")

print(df[["id", "gt_status", "pred_status", "match", "label"]].head(20))
print()
print("Label distribution (per id):")
print(df.drop_duplicates("id")["label"].value_counts(dropna=False).sort_index())

    id            gt_status    pred_status match  label
0   18             Accepted  Runtime Error  diff      3
1   20         Wrong Answer   Wrong Answer   sim      2
2   21        Runtime Error   Wrong Answer  diff      2
3   23         Wrong Answer   Wrong Answer   sim      1
4   29         Wrong Answer  Runtime Error  diff      3
5   35             Accepted  Runtime Error  diff      2
6   36        Runtime Error   Wrong Answer  diff      3
7   46  Time Limit Exceeded       Accepted  diff      3
8   53         Wrong Answer   Wrong Answer   sim      1
9   54        Runtime Error   Wrong Answer  diff      3
10  57             Accepted   Wrong Answer  diff      3
11  60         Wrong Answer   Wrong Answer   sim      1
12  61         Wrong Answer  Runtime Error  diff      3
13  63         Wrong Answer   Wrong Answer   sim      1
14  66         Wrong Answer   Wrong Answer   sim      1
15  69         Wrong Answer   Wrong Answer   sim      1
16  70             Accepted       Accepted   sim

## SAVE OUTPUT

In [7]:
df_out = df.drop_duplicates("id").reset_index(drop=True)
df_out.to_csv(OUT_PATH, index=False)
print("Saved", df_out.shape[0], "ids to", OUT_PATH)

Saved 51 ids to classify3_reference1_qwen3.7-max.csv


In [8]:
FileLink(OUT_PATH)

/kaggle/working/classify3_reference1_qwen3.7-max.csv